# Séance 5 : Approche Stochastique et ALM
## Codes Python du Cours

**Cours:** Comptabilité Prudentielle en Assurance  
**Instructeur:** Fabrice BOREL-MATHURIN  
**Institution:** ACPR-DEAR & SAF Lyon 1  
**Date:** Novembre 2025

---

Ce notebook regroupe l'ensemble des codes Python présentés dans la Séance 5, avec explications et résultats.

## Imports Nécessaires

Importons les bibliothèques indispensables pour tous les exercices.

In [ ]:
!pip install scipy seaborn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Configuration matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ Imports réussis")

---

## PARTIE 1 : SIMULATION VASICEK (Slide 24)

### Code de base : Simulation d'une trajectoire de taux avec le modèle de Vasicek

**Modèle:** $dr_t = a(b - r_t)dt + \sigma dW_t$

**Paramètres:**
- $a = 0.3$ : vitesse de retour à la moyenne
- $b = 0.03$ : taux long terme cible (3%)
- $\sigma = 0.01$ : volatilité
- $r_0 = 0.02$ : taux initial (2%)

In [ ]:
# ==============================================================
# CODE 1 : SIMULATION VASICEK - UNE TRAJECTOIRE
# ==============================================================

# Paramètres Vasicek
a = 0.3           # Vitesse retour à la moyenne
b = 0.03          # Taux long terme (3%)
sigma = 0.01      # Volatilité
r0 = 0.02         # Taux initial (2%)
T = 40            # Horizon (années)
dt = 1/12         # Pas mensuel
N = int(T/dt)     # Nombre de pas = 480 mois

# Simulation d'une trajectoire
np.random.seed(42)  # Pour reproductibilité
r = np.zeros(N+1)
r[0] = r0

for t in range(N):
    Z = np.random.normal(0, 1)  # Tirage aléatoire gaussien
    r[t+1] = r[t] + a*(b - r[t])*dt + sigma*np.sqrt(dt)*Z

# Conversion en années pour l'axe X
temps_annees = np.arange(N+1) * dt

# Visualisation
plt.figure(figsize=(12, 5))
plt.plot(temps_annees, r*100, linewidth=2, color='steelblue', label='Taux court')
plt.axhline(y=b*100, color='red', linestyle='--', linewidth=2, label=f'Taux cible = {b*100}%')
plt.xlabel('Années', fontsize=12)
plt.ylabel('Taux court (%)', fontsize=12)
plt.title('Simulation Vasicek - 1 Trajectoire (40 ans)', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n📊 Statistiques de la trajectoire:")
print(f"  Taux initial: {r[0]*100:.2f}%")
print(f"  Taux final: {r[-1]*100:.2f}%")
print(f"  Taux moyen: {np.mean(r)*100:.2f}%")
print(f"  Écart-type: {np.std(r)*100:.2f}%")
print(f"  Min: {np.min(r)*100:.2f}%, Max: {np.max(r)*100:.2f}%")

### Simulation de multiples trajectoires (N scénarios)

In [ ]:
# ==============================================================
# CODE 2 : SIMULATION VASICEK - N TRAJECTOIRES
# ==============================================================

N_scenarios = 1000  # Nombre de scénarios

# Matrice pour stocker les résultats (scenarios × temps)
r_scenarios = np.zeros((N_scenarios, N+1))
r_scenarios[:, 0] = r0  # Initialisation

# Simulation de N scénarios
np.random.seed(42)
for scenario in range(N_scenarios):
    for t in range(N):
        Z = np.random.normal(0, 1)
        r_scenarios[scenario, t+1] = (r_scenarios[scenario, t] + 
                                       a*(b - r_scenarios[scenario, t])*dt + 
                                       sigma*np.sqrt(dt)*Z)

# Statistiques à chaque point temporel
moyenne = np.mean(r_scenarios, axis=0)
perc5 = np.percentile(r_scenarios, 5, axis=0)   # Percentile 5%
perc95 = np.percentile(r_scenarios, 95, axis=0) # Percentile 95%

# Visualisation
plt.figure(figsize=(14, 6))
# Tracer quelques trajectoires
for i in range(min(50, N_scenarios)):
    plt.plot(temps_annees, r_scenarios[i, :]*100, alpha=0.1, color='steelblue')

# Tracer moyenne et bandes
plt.plot(temps_annees, moyenne*100, color='red', linewidth=3, label='Moyenne')
plt.fill_between(temps_annees, perc5*100, perc95*100, alpha=0.3, color='orange', 
                 label='Intervalle [5%, 95%]')
plt.axhline(y=b*100, color='darkgreen', linestyle='--', linewidth=2, label=f'Taux cible')

plt.xlabel('Années', fontsize=12)
plt.ylabel('Taux court (%)', fontsize=12)
plt.title(f'Simulation Vasicek - {N_scenarios} Scénarios (40 ans)', fontsize=14, fontweight='bold')
plt.legend(fontsize=11, loc='best')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n📈 Résumé des {N_scenarios} scénarios:")
print(f"  Taux moyen final (année 40): {moyenne[-1]*100:.2f}%")
print(f"  Intervalle 5%-95% final: [{perc5[-1]*100:.2f}%, {perc95[-1]*100:.2f}%]")

---

## PARTIE 2 : MODÈLES D'ACTIONS - BLACK-SCHOLES

### Code : Simulation de rendements d'actions

**Modèle:** $dS_t = \mu S_t dt + \sigma S_t dW_t$

**Solution explicite:** $S_t = S_0 \exp\left[(\mu - \frac{\sigma^2}{2})t + \sigma W_t\right]$

In [ ]:
# ==============================================================
# CODE 3 : SIMULATION BLACK-SCHOLES (ACTIONS)
# ==============================================================

# Paramètres actions
S0 = 100          # Prix initial
mu = 0.07         # Rendement moyen (7%)
sigma_action = 0.20  # Volatilité (20%)
T_action = 40     # Horizon (ans)
dt_action = 1/12  # Pas mensuel
N_action = int(T_action/dt_action)

# Simulation d'actions
N_scen_action = 1000
S_scenarios = np.zeros((N_scen_action, N_action+1))
S_scenarios[:, 0] = S0

np.random.seed(42)
for scenario in range(N_scen_action):
    for t in range(N_action):
        Z = np.random.normal(0, 1)
        S_scenarios[scenario, t+1] = (S_scenarios[scenario, t] * 
                                       np.exp((mu - sigma_action**2/2)*dt_action + 
                                              sigma_action*np.sqrt(dt_action)*Z))

# Statistiques
S_moyenne = np.mean(S_scenarios, axis=0)
S_perc5 = np.percentile(S_scenarios, 5, axis=0)
S_perc95 = np.percentile(S_scenarios, 95, axis=0)

# Rendements annuels
rendements = np.diff(S_scenarios, axis=1) / S_scenarios[:, :-1]
rendements_annuels = (1 + rendements.reshape(N_scen_action, -1, 12)).prod(axis=2) - 1

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Trajectoires de prix
ax1 = axes[0]
temps_action = np.arange(N_action+1) * dt_action
for i in range(min(50, N_scen_action)):
    ax1.plot(temps_action, S_scenarios[i, :], alpha=0.1, color='steelblue')
ax1.plot(temps_action, S_moyenne, color='red', linewidth=3, label='Moyenne')
ax1.fill_between(temps_action, S_perc5, S_perc95, alpha=0.3, color='orange')
ax1.set_xlabel('Années')
ax1.set_ylabel('Prix de l\'action')
ax1.set_title('Simulation Black-Scholes - Prix des Actions')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Distribution des rendements annuels (année 1)
ax2 = axes[1]
ax2.hist(rendements_annuels[:, 0]*100, bins=50, alpha=0.7, color='steelblue', edgecolor='black')
ax2.axvline(x=mu*100, color='red', linestyle='--', linewidth=2, label=f'μ = {mu*100}%')
ax2.set_xlabel('Rendement annuel (%)')
ax2.set_ylabel('Fréquence')
ax2.set_title('Distribution des rendements annuels (Année 1)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n📊 Statistiques Actions:")
print(f"  Prix initial: {S0:.2f}€")
print(f"  Prix moyen final (40 ans): {S_moyenne[-1]:.2f}€")
print(f"  Intervalle 5%-95% final: [{S_perc5[-1]:.2f}€, {S_perc95[-1]:.2f}€]")
print(f"  Rendement annuel moyen: {rendements_annuels.mean()*100:.2f}%")

---

## PARTIE 3 : VALORISATION STOCHASTIQUE - CONTRAT EURO AVEC GARANTIE

### Projection d'un contrat avec TMG et participation aux bénéfices

In [ ]:
# ==============================================================
# CODE 4 : PROJECTION CONTRAT EURO AVEC TMG
# ==============================================================

def projeter_contrat_euro(taux_scenarios, rendement_actions_scenarios, 
                          PM0=100, TMG=0.015, pct_PB=0.85, duree=10, 
                          pct_oblig=0.6, pct_action=0.4):
    """
    Projette un contrat euro avec garantie plancher (TMG).
    
    Paramètres:
    -----------
    taux_scenarios : array (N_scenarios, duree)
        Taux d'intérêt pour chaque scénario et année
    rendement_actions_scenarios : array (N_scenarios, duree)
        Rendements actions pour chaque scénario et année
    PM0 : float
        Provision mathématique initiale (en k€)
    TMG : float
        Taux Minimum Garanti (ex: 0.015 = 1.5%)
    pct_PB : float
        Pourcentage de participation aux bénéfices (ex: 0.85 = 85%)
    duree : int
        Durée du contrat en années
    pct_oblig : float
        % allocation obligations (couverture stable)
    pct_action : float
        % allocation actions (rendement variable)
    
    Retourne:
    ---------
    PM_final : array (N_scenarios,)
        PM finale pour chaque scénario
    flux_scenarios : array (N_scenarios, duree)
        Flux (revalorisation) pour chaque scénario
    """
    N_scen = taux_scenarios.shape[0]
    
    PM = np.full(N_scen, PM0)  # PM initiale pour tous les scénarios
    flux_scenarios = np.zeros((N_scen, duree))
    
    for annee in range(duree):
        # Rendement actif = mixte obligations + actions
        rend_actifs = (pct_oblig * taux_scenarios[:, annee] + 
                      pct_action * rendement_actions_scenarios[:, annee])
        
        # Revalorisation garantie
        revalorisation = np.maximum(rend_actifs, TMG)  # Au moins TMG
        
        # Participation aux bénéfices
        surplus_rendement = np.maximum(rend_actifs - TMG, 0)
        PB_distribuee = pct_PB * surplus_rendement
        
        revalorisation_totale = TMG + PB_distribuee
        
        # Mise à jour PM
        PM = PM * (1 + revalorisation_totale)
        flux_scenarios[:, annee] = PM.copy()
    
    PM_final = PM
    return PM_final, flux_scenarios

print("✓ Fonction projeter_contrat_euro définie")

### Calcul du Best Estimate Stochastique

In [ ]:
# ==============================================================
# CODE 5 : CALCUL BE STOCHASTIQUE
# ==============================================================

def calculer_BE_stochastique(PM_final, taux_scenarios, duree=10, 
                             taux_actualisation_marche=None):
    """
    Calcule le Best Estimate stochastique = moyenne des BE sur tous scénarios.
    
    Paramètres:
    -----------
    PM_final : array (N_scenarios,)
        PM finale pour chaque scénario
    taux_scenarios : array (N_scenarios, duree)
        Courbes de taux pour actualisation
    duree : int
        Durée du contrat
    taux_actualisation_marche : float, optional
        Taux déterministe de comparaison (approche non-stochastique)
    
    Retourne:
    ---------
    BE_stoch : float
        Best Estimate stochastique
    BE_deterministe : float
        Best Estimate déterministe (si fourni)
    distribution_BE : array
        BE pour chaque scénario
    """
    N_scen = len(PM_final)
    
    # Calcul du taux d'actualisation moyen par année
    taux_moy = np.mean(taux_scenarios, axis=0)
    
    # Actualisation : chaque scénario avec sa propre courbe de taux
    BE_scenarios = np.zeros(N_scen)
    for scen in range(N_scen):
        # Facteur d'actualisation = produit (1 + taux_annuel)
        facteur_actualisation = np.prod(1 + taux_scenarios[scen, :])
        BE_scenarios[scen] = PM_final[scen] / facteur_actualisation
    
    # BE stochastique = moyenne
    BE_stoch = np.mean(BE_scenarios)
    
    # BE déterministe (optionnel, avec taux moyen)
    BE_deterministe = None
    if taux_actualisation_marche is not None:
        facteur_det = (1 + taux_actualisation_marche) ** duree
        BE_deterministe = np.mean(PM_final) / facteur_det
    
    return BE_stoch, BE_deterministe, BE_scenarios

print("✓ Fonction calculer_BE_stochastique définie")

### Application : Calcul BE pour contrat Euro

In [ ]:
# ==============================================================
# CODE 6 : APPLICATION - BE CONTRAT EURO
# ==============================================================

# Paramètres contrat
PM0 = 100         # Provision mathématique initiale (k€)
TMG = 0.015       # Taux minimum garanti (1.5%)
duree = 10        # Durée (ans)
pct_PB = 0.85     # Participation aux bénéfices (85%)
N_scenarios = 1000  # Nombre de scénarios

# Récupérer les scénarios de taux (déjà générés plus haut)
# Rééchantillonner si nécessaire pour duree=10
indices_10ans = np.arange(0, duree*12+1, 12)  # Prendre tous les 12 mois
taux_10ans = r_scenarios[:N_scenarios, indices_10ans]

# Générer rendements actions
rendements_actions = np.zeros((N_scenarios, duree))
np.random.seed(42)
for scen in range(N_scenarios):
    for an in range(duree):
        # Rendement actions = log-normal
        Z = np.random.normal(0, 1)
        rendements_actions[scen, an] = np.exp((mu - sigma_action**2/2) + 
                                               sigma_action*Z) - 1

# Projection du contrat
PM_final, flux_projetes = projeter_contrat_euro(
    taux_10ans,
    rendements_actions,
    PM0=PM0,
    TMG=TMG,
    pct_PB=pct_PB,
    duree=duree
)

# Calcul BE stochastique
BE_stoch, BE_det, BE_scenarios = calculer_BE_stochastique(
    PM_final,
    taux_10ans,
    duree=duree,
    taux_actualisation_marche=0.02
)

print("\n" + "="*60)
print("📊 RÉSULTATS - CONTRAT EURO AVEC GARANTIE")
print("="*60)
print(f"\nParamètres du contrat:")
print(f"  PM initiale: {PM0} k€")
print(f"  TMG: {TMG*100}%")
print(f"  Participation aux bénéfices: {pct_PB*100}%")
print(f"  Durée: {duree} ans")
print(f"  Nombre de scénarios: {N_scenarios}")

print(f"\nRésultats - PM finale:")
print(f"  Moyenne: {np.mean(PM_final):.2f} k€")
print(f"  Écart-type: {np.std(PM_final):.2f} k€")
print(f"  Min: {np.min(PM_final):.2f} k€")
print(f"  Max: {np.max(PM_final):.2f} k€")

print(f"\nRésultats - Best Estimate:")
print(f"  BE Stochastique: {BE_stoch:.2f} k€")
print(f"  BE Déterministe (2%): {BE_det:.2f} k€")
print(f"  Différence (Sto - Dét): {BE_stoch - BE_det:.2f} k€ ({(BE_stoch-BE_det)/BE_det*100:.1f}%)")
print(f"  (Impact garantie plancher)")

### Visualisation de la distribution du BE

In [ ]:
# ==============================================================
# CODE 7 : VISUALISATION DISTRIBUTION BE
# ==============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Histogramme BE
ax1 = axes[0, 0]
ax1.hist(BE_scenarios, bins=50, alpha=0.7, color='steelblue', edgecolor='black')
ax1.axvline(x=np.mean(BE_scenarios), color='red', linestyle='--', linewidth=2, 
           label=f'Moyenne = {np.mean(BE_scenarios):.2f}')
ax1.axvline(x=BE_det, color='green', linestyle='--', linewidth=2, 
           label=f'Déterministe = {BE_det:.2f}')
ax1.set_xlabel('Best Estimate (k€)')
ax1.set_ylabel('Fréquence')
ax1.set_title('Distribution du BE (1000 scénarios)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. Boîte à moustaches
ax2 = axes[0, 1]
ax2.boxplot(BE_scenarios, vert=True)
ax2.set_ylabel('BE (k€)')
ax2.set_title('Boîte à moustaches - BE')
ax2.grid(True, alpha=0.3)

# 3. Évolution PM par année (tracer quelques scénarios)
ax3 = axes[1, 0]
années = np.arange(duree+1)
for i in range(min(100, N_scenarios)):
    PM_par_année = np.insert(flux_projetes[i, :], 0, PM0)
    ax3.plot(années, PM_par_année, alpha=0.1, color='steelblue')

# Moyenne et quantiles
PM_par_année_moy = np.insert(np.mean(flux_projetes, axis=0), 0, PM0)
PM_par_année_p5 = np.percentile(flux_projetes, 5, axis=0)
PM_par_année_p95 = np.percentile(flux_projetes, 95, axis=0)

ax3.plot(années, PM_par_année_moy, color='red', linewidth=3, label='Moyenne')
ax3.fill_between(années, np.insert(PM_par_année_p5, 0, PM0), 
                 np.insert(PM_par_année_p95, 0, PM0), alpha=0.3, color='orange', 
                 label='[5%, 95%]')
ax3.set_xlabel('Années')
ax3.set_ylabel('PM (k€)')
ax3.set_title('Évolution de la PM - Contrat Euro')
ax3.legend()
ax3.grid(True, alpha=0.3)

# 4. Quantiles en fonction du temps (quantile-quantile plot pour année 10)
ax4 = axes[1, 1]
quantiles = np.percentile(BE_scenarios, [1, 5, 10, 25, 50, 75, 90, 95, 99])
quantiles_labels = ['1%', '5%', '10%', '25%', '50%', '75%', '90%', '95%', '99%']
ax4.barh(quantiles_labels, quantiles, color='steelblue', edgecolor='black')
ax4.axvline(x=np.mean(BE_scenarios), color='red', linestyle='--', linewidth=2, label='Moyenne')
ax4.set_xlabel('BE (k€)')
ax4.set_title('Quantiles du BE')
ax4.legend()
ax4.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print(f"\n📈 Statistiques détaillées du BE:")
print(f"  Quantile 1%: {np.percentile(BE_scenarios, 1):.2f} k€")
print(f"  Quantile 5%: {np.percentile(BE_scenarios, 5):.2f} k€")
print(f"  Quantile 50% (Médiane): {np.percentile(BE_scenarios, 50):.2f} k€")
print(f"  Quantile 95%: {np.percentile(BE_scenarios, 95):.2f} k€")
print(f"  Quantile 99%: {np.percentile(BE_scenarios, 99):.2f} k€")
print(f"  Asymétrie (skewness): {(np.mean(BE_scenarios) - np.percentile(BE_scenarios, 50))/(np.percentile(BE_scenarios, 95) - np.percentile(BE_scenarios, 5)):.3f}")

---

## PARTIE 4 : MANAGEMENT ACTIONS - DÉCISIONS CONDITIONNELLES

### Implémentation avec allocation dynamique d'actifs et PB variable

In [ ]:
# ==============================================================
# CODE 8 : CONTRAT AVEC MANAGEMENT ACTIONS
# ==============================================================

def projeter_contrat_avec_management_actions(taux_scenarios, rendement_actions_scenarios,
                                             PM0=100, TMG=0.015, duree=10):
    """
    Projection contrat euro avec management actions dynamiques.
    
    Management actions implémentées:
    1. Allocation d'actifs : si rendement faible → action = 20%, sinon 40%
    2. Taux de PB : si réserves élevées → PB=90%, sinon PB=70%
    """
    N_scen = taux_scenarios.shape[0]
    
    PM = np.full(N_scen, PM0)
    PPB = np.zeros(N_scen)  # Provision pour PB différée
    flux_projetes = np.zeros((N_scen, duree))
    allocation_actions = np.zeros((N_scen, duree))  # Pour suivi
    taux_PB = np.zeros((N_scen, duree))  # Pour suivi
    
    for annee in range(duree):
        for scen in range(N_scen):
            # 1. Management Action : Allocation dynamique
            rend_previsionnel = (0.6 * taux_scenarios[scen, annee] + 
                                0.4 * rendement_actions_scenarios[scen, annee])
            
            if rend_previsionnel < TMG:
                pct_action = 0.20  # Réduire exposition aux actions
            else:
                pct_action = 0.40  # Normal
            
            allocation_actions[scen, annee] = pct_action
            
            # Rendement actif réel
            rend_actifs = ((1 - pct_action) * taux_scenarios[scen, annee] + 
                          pct_action * rendement_actions_scenarios[scen, annee])
            
            # 2. Management Action : Taux PB variable selon réserves
            if PPB[scen] / (PM[scen] + 1e-6) > 0.15:  # Si PPB/PM > 15%
                pct_PB_distrib = 0.90  # Distribuer généreusement
            else:
                pct_PB_distrib = 0.70  # Moins généreux
            
            taux_PB[scen, annee] = pct_PB_distrib
            
            # Revalorisation
            if rend_actifs < TMG:
                revalorisation = TMG
                cout_garantie = PM[scen] * (TMG - rend_actifs)
                PPB[scen] -= cout_garantie  # Prélèvement sur PPB
            else:
                surplus = rend_actifs - TMG
                revalorisation = TMG + pct_PB_distrib * surplus
                PPB[scen] += (1 - pct_PB_distrib) * surplus  # Accumulation
            
            # Mise à jour PM
            PM[scen] *= (1 + revalorisation)
            flux_projetes[scen, annee] = PM[scen]
    
    return PM, PPB, flux_projetes, allocation_actions, taux_PB

print("✓ Fonction projeter_contrat_avec_management_actions définie")

### Application et comparaison avec/sans management actions

In [ ]:
# ==============================================================
# CODE 9 : COMPARAISON AVEC/SANS MANAGEMENT ACTIONS
# ==============================================================

# Projection SANS management actions (allocation fixe 60/40)
PM_sans_MA, flux_sans_MA = projeter_contrat_euro(
    taux_10ans,
    rendements_actions,
    PM0=PM0,
    TMG=TMG,
    pct_PB=0.75,  # PB fixe
    duree=duree,
    pct_oblig=0.6,
    pct_action=0.4
)

# Projection AVEC management actions
PM_avec_MA, PPB, flux_avec_MA, alloc_actions, taux_pb = projeter_contrat_avec_management_actions(
    taux_10ans,
    rendements_actions,
    PM0=PM0,
    TMG=TMG,
    duree=duree
)

# BE stochastique
BE_stoch_sans, _, BE_scen_sans = calculer_BE_stochastique(PM_sans_MA, taux_10ans, duree=duree)
BE_stoch_avec, _, BE_scen_avec = calculer_BE_stochastique(PM_avec_MA, taux_10ans, duree=duree)

print("\n" + "="*70)
print("🔄 COMPARAISON : MANAGEMENT ACTIONS OUI/NON")
print("="*70)
print(f"\nSans Management Actions (allocation fixe 60/40):")
print(f"  PM finale moyenne: {np.mean(PM_sans_MA):.2f} k€")
print(f"  BE stochastique: {BE_stoch_sans:.2f} k€")
print(f"  Écart-type PM: {np.std(PM_sans_MA):.2f} k€")

print(f"\nAvec Management Actions (allocation dynamique + PB variable):")
print(f"  PM finale moyenne: {np.mean(PM_avec_MA):.2f} k€")
print(f"  BE stochastique: {BE_stoch_avec:.2f} k€")
print(f"  Écart-type PM: {np.std(PM_avec_MA):.2f} k€")
print(f"  PPB moyenne finale: {np.mean(PPB):.2f} k€")

print(f"\nImpact Management Actions:")
print(f"  Δ BE: {BE_stoch_avec - BE_stoch_sans:+.2f} k€ ({(BE_stoch_avec - BE_stoch_sans)/BE_stoch_sans*100:+.1f}%)")
print(f"  Réduction risque (écart-type): {np.std(PM_sans_MA) - np.std(PM_avec_MA):+.2f} k€")

### Visualisation de l'impact des management actions

In [ ]:
# ==============================================================
# CODE 10 : VISUALISATION MANAGEMENT ACTIONS
# ==============================================================

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Comparaison distribution BE
ax1 = axes[0, 0]
ax1.hist(BE_scen_sans, bins=40, alpha=0.6, label='Sans MA', color='coral', edgecolor='black')
ax1.hist(BE_scen_avec, bins=40, alpha=0.6, label='Avec MA', color='steelblue', edgecolor='black')
ax1.axvline(x=BE_stoch_sans, color='red', linestyle='--', linewidth=2)
ax1.axvline(x=BE_stoch_avec, color='darkblue', linestyle='--', linewidth=2)
ax1.set_xlabel('BE (k€)')
ax1.set_ylabel('Fréquence')
ax1.set_title('Distribution du BE : Avec vs Sans Management Actions')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. Allocation d'actions moyenne par année
ax2 = axes[0, 1]
années = np.arange(1, duree+1)
alloc_moy = np.mean(alloc_actions, axis=0)
ax2.bar(années, alloc_moy*100, color='steelblue', alpha=0.7, edgecolor='black')
ax2.axhline(y=40, color='red', linestyle='--', linewidth=2, label='Allocation normale (40%)')
ax2.axhline(y=20, color='orange', linestyle='--', linewidth=2, label='Allocation réduite (20%)')
ax2.set_xlabel('Année')
ax2.set_ylabel('% Actions')
ax2.set_title('Allocation d\'actions moyenne par année')
ax2.legend()
ax2.grid(True, alpha=0.3, axis='y')

# 3. Taux de PB moyen par année
ax3 = axes[1, 0]
taux_pb_moy = np.mean(taux_pb, axis=0)
ax3.plot(années, taux_pb_moy*100, marker='o', linewidth=2, markersize=8, color='steelblue')
ax3.axhline(y=75, color='gray', linestyle='--', linewidth=1, alpha=0.5, label='Taux nominal')
ax3.fill_between(années, 70, 90, alpha=0.1, color='orange', label='Intervalle MA')
ax3.set_xlabel('Année')
ax3.set_ylabel('Taux de PB (%)')
ax3.set_title('Taux de Participation aux Bénéfices moyen')
ax3.legend()
ax3.grid(True, alpha=0.3)
ax3.set_ylim(65, 95)

# 4. Evolution PM : Sans vs Avec MA
ax4 = axes[1, 1]
années_proj = np.arange(duree+1)
PM_sans_moy = np.insert(np.mean(flux_sans_MA, axis=0), 0, PM0)
PM_avec_moy = np.insert(np.mean(flux_avec_MA, axis=0), 0, PM0)

ax4.plot(années_proj, PM_sans_moy, marker='s', linewidth=2, markersize=6, 
        label='Sans MA', color='coral')
ax4.plot(années_proj, PM_avec_moy, marker='o', linewidth=2, markersize=6, 
        label='Avec MA', color='steelblue')
ax4.set_xlabel('Année')
ax4.set_ylabel('PM (k€)')
ax4.set_title('Evolution PM moyenne')
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## PARTIE 5 : ASSET-LIABILITY MANAGEMENT (ALM)

### Calcul de Duration et Duration Gap

In [ ]:
# ==============================================================
# CODE 11 : CALCUL DURATION
# ==============================================================

def calculer_duration(flux_paiements, temps, taux_actualisation):
    """
    Calcule la duration d'un portefeuille de flux.
    
    Duration = Σ(t × VA(flux_t)) / Σ(VA(flux_t))
    où VA = Valeur Actualisée
    
    Paramètres:
    -----------
    flux_paiements : array
        Flux de paiements (ex: coupons, remboursements)
    temps : array
        Temps des flux (en années)
    taux_actualisation : float
        Taux utilisé pour l'actualisation
    """
    # Actualisation
    VA_flux = flux_paiements / (1 + taux_actualisation) ** temps
    
    # Duration
    duration = np.sum(temps * VA_flux) / np.sum(VA_flux)
    
    # Valeur actualisée totale
    valeur_totale = np.sum(VA_flux)
    
    return duration, valeur_totale

# Exemple : Rente viagère (10 ans)
temps_rente = np.arange(1, 11)  # 1, 2, ..., 10 ans
flux_rente = np.full(10, 10)  # 10 k€ par an
taux_actualisation = 0.02  # 2%

D_passif, VA_passif = calculer_duration(flux_rente, temps_rente, taux_actualisation)

print("\n" + "="*60)
print("📊 EXAMPLE ALM - CALCUL DURATION")
print("="*60)
print(f"\nPASSIF (Rente viagère):")
print(f"  Flux: {flux_rente} k€/an pendant 10 ans")
print(f"  Taux actualisation: {taux_actualisation*100}%")
print(f"  Valeur Actualisée: {VA_passif:.2f} k€")
print(f"  Duration: {D_passif:.2f} ans")

# Exemple : Portefeuille d'obligations
# Obligation 1 : Maturité 5 ans, Coupon 2%, Valeur nominale 100
# Obligation 2 : Maturité 10 ans, Coupon 3%, Valeur nominale 100

# Flux obligation 1
flux_ob1 = np.array([2, 2, 2, 2, 102])  # Coupons + remboursement
temps_ob1 = np.array([1, 2, 3, 4, 5])

# Flux obligation 2
flux_ob2 = np.array([3, 3, 3, 3, 3, 3, 3, 3, 3, 103])  # Coupons + remboursement
temps_ob2 = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])

D_ob1, VA_ob1 = calculer_duration(flux_ob1, temps_ob1, taux_actualisation)
D_ob2, VA_ob2 = calculer_duration(flux_ob2, temps_ob2, taux_actualisation)

# Portefeuille : 50 unités obligation 1, 50 unités obligation 2
VA_actif = 50 * VA_ob1 + 50 * VA_ob2
D_actif = (50 * VA_ob1 * D_ob1 + 50 * VA_ob2 * D_ob2) / VA_actif

print(f"\nACTIF (Portefeuille obligations):")
print(f"  Obligation 1 (5 ans, 2% coupon):")
print(f"    VA: {VA_ob1:.2f} k€, Duration: {D_ob1:.2f} ans")
print(f"  Obligation 2 (10 ans, 3% coupon):")
print(f"    VA: {VA_ob2:.2f} k€, Duration: {D_ob2:.2f} ans")
print(f"  Portefeuille (50 + 50):")
print(f"    VA: {VA_actif:.2f} k€")
print(f"    Duration: {D_actif:.2f} ans")

# Duration Gap
gap = D_actif - D_passif
print(f"\n{'='*60}")
print(f"DURATION GAP = {D_actif:.2f} - {D_passif:.2f} = {gap:.2f} ans")
if gap < 0:
    print(f"⚠️  Gap NÉGATIF: risque de RÉINVESTISSEMENT")
    print(f"   → Actifs plus courts que passifs")
    print(f"   → Action: allonger la duration de l'actif")
elif gap > 0:
    print(f"⚠️  Gap POSITIF: risque de TAUX À LA HAUSSE")
    print(f"   → Actifs plus longs que passifs")
    print(f"   → Action: raccourcir la duration de l'actif")
else:
    print(f"✓ Gap NUL: IMMUNISATION parfaite")

### Impact d'un choc de taux sur l'ALM

In [ ]:
# ==============================================================
# CODE 12 : IMPACT CHOC DE TAUX SUR ALM
# ==============================================================

def analyser_sensibilite_taux(VA_actif, D_actif, VA_passif, D_passif, 
                              chocs_taux):
    """
    Analyse l'impact d'un choc de taux sur le ratio de couverture.
    
    Utilise l'approximation linéaire :
    ΔVA ≈ -Duration × Δtaux × VA
    """
    resultats = []
    
    for choc in chocs_taux:
        # Impact sur actif
        delta_VA_actif = -D_actif * choc * VA_actif
        VA_actif_choque = VA_actif + delta_VA_actif
        
        # Impact sur passif
        delta_VA_passif = -D_passif * choc * VA_passif
        VA_passif_choque = VA_passif + delta_VA_passif
        
        # Ratio de couverture
        ratio_avant = VA_actif / VA_passif
        ratio_apres = VA_actif_choque / VA_passif_choque
        
        resultats.append({
            'choc': choc,
            'VA_actif_avant': VA_actif,
            'VA_actif_apres': VA_actif_choque,
            'VA_passif_avant': VA_passif,
            'VA_passif_apres': VA_passif_choque,
            'ratio_avant': ratio_avant,
            'ratio_apres': ratio_apres,
            'delta_ratio': ratio_apres - ratio_avant
        })
    
    return pd.DataFrame(resultats)

# Chocs de taux à analyser
chocs = np.linspace(-0.03, 0.03, 7)  # -300 bp à +300 bp

# Analyse
resultat_alm = analyser_sensibilite_taux(
    VA_actif, D_actif,
    VA_passif, D_passif,
    chocs
)

print("\n" + "="*100)
print("📊 ANALYSE SENSIBILITÉ ALM - IMPACT CHOC DE TAUX")
print("="*100)
print(f"\nParamètres:")
print(f"  VA Actif: {VA_actif:.2f} k€, Duration: {D_actif:.2f} ans")
print(f"  VA Passif: {VA_passif:.2f} k€, Duration: {D_passif:.2f} ans")
print(f"  Duration Gap: {gap:.2f} ans")

print(f"\n" + resultat_alm.to_string(index=False))

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 1. Valeurs actualisées
ax1 = axes[0]
ax1.plot(resultat_alm['choc']*100, resultat_alm['VA_actif_apres'], 
        marker='o', linewidth=2, label='Actif', color='steelblue')
ax1.plot(resultat_alm['choc']*100, resultat_alm['VA_passif_apres'], 
        marker='s', linewidth=2, label='Passif', color='coral')
ax1.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
ax1.set_xlabel('Choc de taux (bp)')
ax1.set_ylabel('Valeur Actualisée (k€)')
ax1.set_title('Impact du choc de taux sur VA')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. Ratio de couverture
ax2 = axes[1]
ax2.plot(resultat_alm['choc']*100, resultat_alm['ratio_apres'], 
        marker='o', linewidth=2.5, color='darkgreen', markersize=8)
ax2.axhline(y=1.0, color='red', linestyle='--', linewidth=2, label='Seuil 100%')
ax2.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
ax2.fill_between(resultat_alm['choc']*100, 0.95, 1.05, alpha=0.2, color='green', label='Zone sûre')
ax2.set_xlabel('Choc de taux (bp)')
ax2.set_ylabel('Ratio de couverture (Actif/Passif)')
ax2.set_title('Ratio de couverture après choc')
ax2.set_ylim(0.85, 1.25)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Optimisation d'allocation pour immunisation

In [ ]:
# ==============================================================
# CODE 13 : OPTIMISATION ALLOCATION POUR IMMUNISATION
# ==============================================================

def optimiser_allocation_ALM(D_actif_long, D_actif_court,
                             VA_passif, D_passif,
                             VA_actif_cible=None):
    """
    Optimise l'allocation entre obligations longues et courtes
    pour minimiser le duration gap.
    
    Variables:
    - x : pourcentage en obligations longues
    - (1-x) : pourcentage en obligations courtes
    
    Contrainte: D_actif_mix = D_passif (immunisation)
    D_actif_mix = x * D_actif_long + (1-x) * D_actif_court
    """
    
    if VA_actif_cible is None:
        VA_actif_cible = VA_passif * 1.05  # Ratio minimum 105%
    
    # Résoudre : x * D_long + (1-x) * D_court = D_passif
    # x * D_long + D_court - x * D_court = D_passif
    # x * (D_long - D_court) = D_passif - D_court
    
    if D_actif_long != D_actif_court:
        x_optimal = (D_passif - D_actif_court) / (D_actif_long - D_actif_court)
        x_optimal = np.clip(x_optimal, 0, 1)  # Constraint: 0 ≤ x ≤ 1
    else:
        x_optimal = 0.5
    
    D_actif_mix = x_optimal * D_actif_long + (1 - x_optimal) * D_actif_court
    
    gap = D_actif_mix - D_passif
    
    return {
        'allocation_long': x_optimal,
        'allocation_court': 1 - x_optimal,
        'D_actif_mix': D_actif_mix,
        'duration_gap': gap,
        'immunise': abs(gap) < 0.1  # Tolérance: 0.1 an
    }

# Données
D_long = 12    # Obligations longues (12 ans)
D_court = 3    # Obligations courtes (3 ans)

# Optimisation
resultat_optim = optimiser_allocation_ALM(
    D_long, D_court,
    VA_passif, D_passif
)

print("\n" + "="*70)
print("🎯 OPTIMISATION ALLOCATION - IMMUNISATION ALM")
print("="*70)
print(f"\nDonnées:")
print(f"  VA Passif: {VA_passif:.2f} k€")
print(f"  Duration Passif: {D_passif:.2f} ans")
print(f"  Duration Obligations Longues: {D_long} ans")
print(f"  Duration Obligations Courtes: {D_court} ans")

print(f"\nAllocation Optimale:")
print(f"  % Obligations Longues: {resultat_optim['allocation_long']*100:.1f}%")
print(f"  % Obligations Courtes: {resultat_optim['allocation_court']*100:.1f}%")

print(f"\nRésultats:")
print(f"  Duration Actif Mix: {resultat_optim['D_actif_mix']:.2f} ans")
print(f"  Duration Gap: {resultat_optim['duration_gap']:.2f} ans")
print(f"  Immunisé: {'✓ OUI' if resultat_optim['immunise'] else '✗ NON'}")

# Représentation graphique
fig, ax = plt.subplots(figsize=(12, 6))

allocations = ['Court\n(3 ans)', 'Optimal\nMix', 'Long\n(12 ans)']
durations = [D_court, resultat_optim['D_actif_mix'], D_long]
colors = ['coral', 'steelblue', 'lightcoral']

bars = ax.bar(allocations, durations, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
ax.axhline(y=D_passif, color='red', linestyle='--', linewidth=3, label=f'Duration Passif = {D_passif:.2f}')

# Annotations
for i, (bar, dur) in enumerate(zip(bars, durations)):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
           f'{dur:.2f} ans',
           ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_ylabel('Duration (ans)', fontsize=12)
ax.set_title('Allocation Optimale pour Immunisation ALM', fontsize=14, fontweight='bold')
ax.set_ylim(0, max(D_long, D_passif) * 1.2)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

---

## SYNTHÈSE ET STATISTIQUES GLOBALES

In [ ]:
# ==============================================================
# CODE 14 : SYNTHÈSE ET STATISTIQUES GLOBALES
# ==============================================================

print("\n" + "="*80)
print("📚 SYNTHÈSE COMPLÈTE - SÉANCE 5 : STOCHASTIQUE & ALM")
print("="*80)

print("\n" + "─"*80)
print("1️⃣  GÉNÉRATEURS DE SCÉNARIOS ÉCONOMIQUES (ESG)")
print("─"*80)
print(f"""
Taux d'intérêt (Vasicek):
  • Modèle: dr_t = a(b - r_t)dt + σ dW_t
  • Paramètres: a={a}, b={b*100}%, σ={sigma*100}%
  • {N_scenarios} scénarios générés
  • Horizon: {T} ans
  • Taux moyen final: {np.mean(r_scenarios[:, -1])*100:.2f}%

Actions (Black-Scholes):
  • Modèle: dS_t = μ S_t dt + σ S_t dW_t
  • Paramètres: μ={mu*100}%, σ={sigma_action*100}%
  • Prix initial: {S0}€
  • Prix moyen final: {np.mean(S_scenarios[:, -1]):.2f}€
""")

print("\n" + "─"*80)
print("2️⃣  VALORISATION STOCHASTIQUE - CONTRAT EURO")
print("─"*80)
print(f"""
Caractéristiques:
  • PM initiale: {PM0} k€
  • TMG: {TMG*100}%
  • Durée: {duree} ans
  • Allocation: 60% obligations, 40% actions

Résultats (sans Management Actions):
  • BE Stochastique: {BE_stoch:.2f} k€
  • BE Déterministe: {BE_det:.2f} k€
  • Écart (impact garantie): {BE_stoch - BE_det:+.2f} k€ ({(BE_stoch-BE_det)/BE_det*100:+.1f}%)
  • Écart-type BE: {np.std(BE_scenarios):.2f} k€
  • Quantile 95%: {np.percentile(BE_scenarios, 95):.2f} k€
""")

print("\n" + "─"*80)
print("3️⃣  MANAGEMENT ACTIONS")
print("─"*80)
print(f"""
Implémentation:
  • Allocation dynamique: 20% (faible) à 40% (normal)
  • Taux PB variable: 70% à 90% selon réserves

Impact sur BE:
  • BE sans MA: {BE_stoch_sans:.2f} k€
  • BE avec MA: {BE_stoch_avec:.2f} k€
  • Amélioration: {BE_stoch_avec - BE_stoch_sans:+.2f} k€
  • Réduction risque (écart-type): {np.std(PM_sans_MA) - np.std(PM_avec_MA):.2f} k€
""")

print("\n" + "─"*80)
print("4️⃣  ALM - IMMUNISATION")
print("─"*80)
print(f"""
Analyse initiale:
  • VA Actif: {VA_actif:.2f} k€, Duration: {D_actif:.2f} ans
  • VA Passif: {VA_passif:.2f} k€, Duration: {D_passif:.2f} ans
  • Duration Gap: {gap:.2f} ans
  • Interprétation: Risque de réinvestissement

Allocation optimale:
  • % Obligations Longues (12 ans): {resultat_optim['allocation_long']*100:.1f}%
  • % Obligations Courtes (3 ans): {resultat_optim['allocation_court']*100:.1f}%
  • Duration résultante: {resultat_optim['D_actif_mix']:.2f} ans
  • New Gap: {resultat_optim['duration_gap']:.2f} ans (Immunisé: {resultat_optim['immunise']})
""")

print("\n" + "─"*80)
print("5️⃣  ANALYSE DE SENSIBILITÉ - CHOC DE TAUX")
print("─"*80)
print(f"""
Impact sur ratio de couverture:
  • À -200 bp: Ratio = {resultat_alm[resultat_alm['choc'] == -0.02]['ratio_apres'].values[0]:.2f}
  • À 0 bp: Ratio = {resultat_alm[resultat_alm['choc'] == 0]['ratio_apres'].values[0]:.2f}
  • À +200 bp: Ratio = {resultat_alm[resultat_alm['choc'] == 0.02]['ratio_apres'].values[0]:.2f}
""")

print("\n" + "="*80)
print("✓ FIN DE LA SYNTHÈSE")
print("="*80)

---

## RESSOURCES ADDITIONNELLES

### Points clés à retenir

1. **Limites de l'approche déterministe**:
   - Sous-estimation des options et garanties
   - Absence de prise en compte des management actions
   - Modélisation insuffisante du comportement dynamique des assurés

2. **Approche stochastique**:
   - Génération de 1 000 à 10 000 scénarios économiques
   - Monte-Carlo pour calcul du BE stochastique
   - Capture la distribution complète des futurs possibles

3. **Management Actions**:
   - Allocation dynamique d'actifs
   - Participation aux bénéfices variable
   - Gestion des rachats
   - Impact significatif sur BE et risque

4. **ALM**:
   - Duration gap comme mesure clé
   - Immunisation pour solvabilité long terme
   - Analyse de sensibilité aux chocs de taux
   - Matching Adjustment pour stabilité prudentielle

### Modèles implémentés

| Élément | Modèle | Utilisation |
|---------|--------|-------------|
| **Taux d'intérêt** | Vasicek, Hull-White | ESG, actualisation |
| **Actions** | Black-Scholes | Rendements portefeuille |
| **BE** | Monte-Carlo | Valorisation stochastique |
| **ALM** | Duration Matching | Immunisation |

### Fichiers Python

Tous les codes ci-dessus sont prêts à l'emploi et peuvent être adaptés pour:
- Différents paramètres de contrats
- Différents horizons de projection
- Intégration dans workflows de production

---

## Contact & Informations

**Instructeur:** Fabrice BOREL-MATHURIN  
**Institutions:** ACPR-DEAR & SAF Université Claude Bernard Lyon 1  
**Date:** Novembre 2025  

Pour toute question ou clarification, consultez la présentation complète ou les ressources additionnelles fournis.